# AgroDatos · Bases de datos relacionales con MySQL en Google Colab

**Curso:** Fundamentos de bases de datos para desarrollo web
**Contexto del curso:** una empresa agrícola colombiana necesita registrar sus fincas, lotes, insumos (fertilizantes, fungicidas, insecticidas) y las detecciones que hace un dron con visión artificial (YOLO). Ustedes van a construir esa base de datos paso a paso.

| # | Ejercicio | Conceptos clave |
|---|-----------|-----------------|
| 0 | Preparar el entorno | Servidor MySQL, cliente Python, conexión |
| 1 | Modelado y creación de tablas (DDL) | Entidad, atributo, tipos de dato, PRIMARY KEY, NOT NULL, CHECK, UNIQUE |
| 2 | Integridad referencial e insumos con ficha técnica | FOREIGN KEY, ON DELETE, INSERT, errores de restricción |
| 3 | Consultas para tomar decisiones | SELECT, WHERE, JOIN, GROUP BY, HAVING, funciones de agregado |
| 4 | Normalización de una planilla de campo | Anomalías, 1FN, 2FN, 3FN, INSERT … SELECT |
| 5 | Visión artificial (YOLO) + MySQL | Guardar detecciones, vistas, triggers, consultas analíticas |

**Cómo trabajar:** ejecuten las celdas en orden (`Shift + Enter`). Las celdas marcadas con ✏️ **RETO** son para que ustedes escriban el SQL. Al final de cada ejercicio hay **preguntas de reflexión** que se responden en la celda de texto indicada.

> Los valores de dosis, carencia y registro de los productos son **didácticos**. En una finca real siempre se consulta la etiqueta y la ficha técnica aprobada por el ICA.

---
## Ejercicio 0 · Preparar el entorno

Google Colab es una máquina Linux (Ubuntu) temporal. Por eso podemos **instalar un servidor MySQL real** dentro de ella. La arquitectura es la misma que usarán en un proyecto web:

```
 [ Su código Python / PHP / Node ]  ──(usuario + contraseña, puerto 3306)──▶  [ Servidor MySQL ]  ──▶  [ Archivos de datos ]
          (cliente)                                                             (motor de BD)
```

- El **servidor** (`mysqld`) guarda los datos y ejecuta las sentencias SQL.
- El **cliente** (aquí la librería `pymysql`) envía las sentencias y recibe los resultados.
- En una página web, el cliente es el *backend* (PHP, Node.js, Python); el navegador nunca se conecta directo a MySQL.

⚠️ La máquina de Colab se borra al cerrar la sesión. Si vuelven otro día, ejecuten de nuevo desde aquí.

In [ ]:
# 0.1 Instalar e iniciar el servidor MySQL (tarda 1 a 2 minutos)
!apt-get -qq update > /dev/null
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mysql-server > /dev/null
!service mysql start

# Crear un usuario para la clase (en producción nunca se trabaja con root)
!mysql -e "CREATE USER IF NOT EXISTS 'alumno'@'localhost' IDENTIFIED WITH mysql_native_password BY 'Colab_2026'; GRANT ALL PRIVILEGES ON *.* TO 'alumno'@'localhost'; FLUSH PRIVILEGES;"
!mysql --version
!pip -q install pymysql

In [ ]:
# 0.2 Funciones de ayuda: sql() ejecuta una o varias sentencias y muestra los resultados como tabla
import pymysql, pandas as pd
from IPython.display import display

CON = pymysql.connect(host="localhost", user="alumno", password="Colab_2026",
                      autocommit=True, charset="utf8mb4")

def _dividir(script):
    # Separa sentencias por ';' respetando el texto entre comillas
    partes, actual, comilla = [], [], None
    for c in script:
        if comilla:
            actual.append(c)
            if c == comilla: comilla = None
        elif c in ("'", '"'):
            comilla = c; actual.append(c)
        elif c == ";":
            partes.append("".join(actual)); actual = []
        else:
            actual.append(c)
    partes.append("".join(actual))
    limpias = []
    for p in partes:
        lineas = [l for l in p.splitlines() if not l.strip().startswith("--")]
        p = "\n".join(lineas).strip()
        if p: limpias.append(p)
    return limpias

def sql(script, mostrar=True):
    # Ejecuta el script. Devuelve el DataFrame de la última consulta que produjo filas.
    CON.ping(reconnect=True)
    ultimo = None
    with CON.cursor() as cur:
        for s in _dividir(script):
            if s.upper().startswith("USE "):
                CON.select_db(s.split()[1].strip("`")); continue
            try:
                cur.execute(s)
            except pymysql.MySQLError as e:
                codigo, mensaje = e.args[0], e.args[1] if len(e.args) > 1 else ""
                print(f"ERROR MySQL {codigo}: {mensaje}\n   en la sentencia: {s[:120]}")
                return None
            if cur.description:
                cols = [d[0] for d in cur.description]
                ultimo = pd.DataFrame(cur.fetchall(), columns=cols)
                if mostrar: display(ultimo)
            elif mostrar and s.split()[0].upper() in ("INSERT", "UPDATE", "DELETE"):
                print(f"OK · {cur.rowcount} fila(s) afectada(s) · {s.split()[0].upper()}")
    return ultimo

def sql_bloque(sentencia):
    # Para TRIGGERS y procedimientos: se envían completos, sin dividir por ';'
    CON.ping(reconnect=True)
    with CON.cursor() as cur:
        cur.execute(sentencia)
    print("OK · bloque ejecutado")

sql("SELECT VERSION() AS version_mysql, NOW() AS hora_servidor")

---
## Ejercicio 1 · Modelado y creación de tablas (DDL)

### Objetivo
Pasar de una descripción del negocio a un **modelo relacional** y crearlo con sentencias DDL (*Data Definition Language*).

### Fundamento
Una base de datos relacional organiza la información en **relaciones** (tablas). Cada fila es una **tupla** y cada columna un **atributo** con un **dominio** (tipo de dato). Las reglas de integridad las hace cumplir el motor, no el programador:

| Restricción | Qué garantiza | Ejemplo en el cultivo |
|---|---|---|
| `PRIMARY KEY` | Cada fila se identifica de forma única y no nula | `id_lote` |
| `NOT NULL` | El dato es obligatorio | Todo lote tiene área |
| `UNIQUE` | No se repite el valor | El código `ESP-L01` no puede existir dos veces |
| `CHECK` | El valor está dentro del dominio permitido | `area_ha > 0` |
| `DEFAULT` | Valor por defecto si no se envía | `estado = 'activo'` |

**Descripción del negocio:** *La empresa tiene varias fincas, cada una ubicada en un municipio y departamento. Cada finca se divide en lotes; un lote tiene un código único, un cultivo sembrado, un área en hectáreas y una fecha de siembra.*

Entidades: **FINCA (1) ──< LOTE (N)**. Por ahora creamos las tablas sin relacionarlas; la llave foránea llega en el Ejercicio 2.

### Tipos de dato en MySQL que usaremos
`INT` enteros · `DECIMAL(p,s)` números exactos (áreas, dinero; nunca `FLOAT` para dinero) · `VARCHAR(n)` texto variable · `DATE` fechas · `ENUM(...)` lista cerrada de valores.

In [ ]:
# 1.1 Crear la base de datos y las dos primeras tablas
sql('''
DROP DATABASE IF EXISTS agro_campo;
CREATE DATABASE agro_campo CHARACTER SET utf8mb4 COLLATE utf8mb4_spanish_ci;
USE agro_campo;

CREATE TABLE finca (
    id_finca      INT AUTO_INCREMENT PRIMARY KEY,
    nombre        VARCHAR(80)  NOT NULL,
    municipio     VARCHAR(60)  NOT NULL,
    departamento  VARCHAR(60)  NOT NULL,
    altitud_msnm  INT CHECK (altitud_msnm BETWEEN 0 AND 4000)
);

CREATE TABLE lote (
    id_lote        INT AUTO_INCREMENT PRIMARY KEY,
    codigo         VARCHAR(10)  NOT NULL UNIQUE,
    id_finca       INT          NOT NULL,
    cultivo        ENUM('maíz','café','papa','tomate','aguacate') NOT NULL,
    area_ha        DECIMAL(6,2) NOT NULL CHECK (area_ha > 0),
    fecha_siembra  DATE,
    estado         VARCHAR(15)  NOT NULL DEFAULT 'activo'
);

SHOW TABLES;
''')

In [ ]:
# 1.2 Revisar la estructura que quedó guardada en el diccionario de datos
sql("DESCRIBE lote")

In [ ]:
# 1.3 Insertar datos de ejemplo
sql('''
INSERT INTO finca (nombre, municipio, departamento, altitud_msnm) VALUES
  ('La Esperanza', 'Fusagasugá', 'Cundinamarca', 1750),
  ('El Porvenir',  'Chinchiná',  'Caldas',       1360),
  ('Villa Rosa',   'Ventaquemada','Boyacá',      2630);

INSERT INTO lote (codigo, id_finca, cultivo, area_ha, fecha_siembra) VALUES
  ('ESP-L01', 1, 'maíz',   3.50, '2026-03-02'),
  ('ESP-L02', 1, 'tomate', 1.20, '2026-05-15'),
  ('POR-L01', 2, 'café',   6.80, '2023-04-10'),
  ('POR-L02', 2, 'maíz',   2.40, '2026-04-20'),
  ('VRO-L01', 3, 'papa',   4.10, '2026-06-01');

SELECT * FROM lote;
''')

### Probemos las restricciones
Cada celda siguiente **debe fallar**. Lean el mensaje de error: es el motor protegiendo la calidad de los datos.

In [ ]:
# 1.4 Violación de CHECK: un lote no puede tener área negativa
sql("INSERT INTO lote (codigo, id_finca, cultivo, area_ha) VALUES ('ESP-L03', 1, 'maíz', -2)")

In [ ]:
# 1.5 Violación de UNIQUE: el código ESP-L01 ya existe
sql("INSERT INTO lote (codigo, id_finca, cultivo, area_ha) VALUES ('ESP-L01', 1, 'papa', 1.0)")

In [ ]:
# 1.6 Violación de ENUM: 'arroz' no está en la lista de cultivos permitidos
sql("INSERT INTO lote (codigo, id_finca, cultivo, area_ha) VALUES ('ESP-L04', 1, 'arroz', 1.0)")

### ✏️ RETO 1
1. Agreguen a `lote` la columna `tipo_riego` con los valores posibles `'goteo'`, `'aspersión'`, `'gravedad'`, `'secano'` y valor por defecto `'secano'` (pista: `ALTER TABLE … ADD COLUMN …`).
2. Creen la tabla `operario` con: `id_operario` (PK autoincremental), `documento` (obligatorio y único), `nombre_completo` (obligatorio), `telefono` y `certificado_aplicador` (`BOOLEAN`, por defecto falso).
3. Inserten 3 operarios.

In [ ]:
# ✏️ Escriban aquí su solución del RETO 1
sql('''
-- ALTER TABLE lote ADD COLUMN ...

-- CREATE TABLE operario ( ... );

-- INSERT INTO operario ...
''')

### Preguntas de reflexión 1
1. ¿Por qué `area_ha` es `DECIMAL(6,2)` y no `FLOAT`? ¿Qué problema de redondeo se evita?
2. ¿Qué diferencia hay entre validar el área en el formulario HTML de la página web y validarla con `CHECK` en MySQL? ¿Por qué conviene hacer ambas?
3. ¿Qué desventaja tiene usar `ENUM` para el cultivo si mañana la empresa empieza a sembrar arroz? ¿Qué alternativa propondrían?

*Respuestas:* (escriban aquí)

---
## Ejercicio 2 · Integridad referencial e insumos con ficha técnica

### Objetivo
Relacionar tablas con **llaves foráneas** y registrar productos agrícolas a partir de su **anexo técnico**, como se hace en la bodega de una finca.

### Fundamento
Una **FOREIGN KEY** obliga a que el valor de una columna exista como llave primaria en otra tabla. Así no puede existir un lote de una finca inexistente ni una aplicación de un producto que no está en el inventario. Al borrar o actualizar la fila *padre* se decide qué pasa con las *hijas*:

| Acción | Efecto al borrar el padre |
|---|---|
| `RESTRICT` / `NO ACTION` | Impide el borrado mientras existan hijos (la opción más segura) |
| `CASCADE` | Borra también los hijos |
| `SET NULL` | Deja la columna del hijo en `NULL` |

### El anexo técnico de un insumo
Cada producto agrícola trae una ficha con datos que la base de datos debe guardar con exactitud, porque de ellos dependen la seguridad del operario y del consumidor:

- **Tipo:** fertilizante, enmienda, fungicida, insecticida, herbicida, bioinsumo.
- **Ingrediente activo** y concentración.
- **Categoría toxicológica:** I (extremadamente tóxico, franja roja), II (altamente tóxico, amarilla), III (medianamente tóxico, azul), IV (ligeramente tóxico, verde).
- **Registro ICA:** número de registro de venta.
- **Dosis por hectárea** y unidad.
- **Periodo de carencia (días):** tiempo mínimo entre la última aplicación y la cosecha.
- **Periodo de reingreso (horas):** tiempo antes de que una persona pueda volver a entrar al lote.

Modelo: **LOTE (1) ──< APLICACION >── (1) PRODUCTO**, es decir, una relación **muchos a muchos** entre lotes y productos resuelta con una tabla intermedia que además guarda fecha y cantidad.

In [ ]:
# 2.1 Relacionar lote con finca (la tabla ya existe, agregamos la restricción)
sql('''
USE agro_campo;
ALTER TABLE lote
  ADD CONSTRAINT fk_lote_finca FOREIGN KEY (id_finca)
  REFERENCES finca(id_finca) ON DELETE RESTRICT ON UPDATE CASCADE;
''')

# 2.2 Tabla de productos con las reglas del anexo técnico
sql('''
CREATE TABLE producto (
    id_producto        INT AUTO_INCREMENT PRIMARY KEY,
    nombre_comercial   VARCHAR(80) NOT NULL UNIQUE,
    tipo               ENUM('fertilizante','enmienda','fungicida','insecticida','herbicida','bioinsumo') NOT NULL,
    ingrediente_activo VARCHAR(120) NOT NULL,
    categoria_tox      ENUM('I','II','III','IV') NOT NULL,
    registro_ica       VARCHAR(20) NOT NULL UNIQUE,
    dosis_ha           DECIMAL(8,2) NOT NULL CHECK (dosis_ha > 0),
    unidad             ENUM('kg','L','g','mL') NOT NULL,
    carencia_dias      INT NOT NULL DEFAULT 0 CHECK (carencia_dias >= 0),
    reingreso_horas    INT NOT NULL DEFAULT 0 CHECK (reingreso_horas >= 0),
    precio_unidad      DECIMAL(10,2) NOT NULL CHECK (precio_unidad >= 0)
);

CREATE TABLE aplicacion (
    id_aplicacion  INT AUTO_INCREMENT PRIMARY KEY,
    id_lote        INT NOT NULL,
    id_producto    INT NOT NULL,
    fecha          DATE NOT NULL,
    cantidad       DECIMAL(8,2) NOT NULL CHECK (cantidad > 0),
    responsable    VARCHAR(80) NOT NULL,
    CONSTRAINT fk_apl_lote     FOREIGN KEY (id_lote)     REFERENCES lote(id_lote)         ON DELETE RESTRICT,
    CONSTRAINT fk_apl_producto FOREIGN KEY (id_producto) REFERENCES producto(id_producto) ON DELETE RESTRICT
);
''')

### Anexos técnicos para registrar (valores didácticos)

| Nombre comercial | Tipo | Ingrediente activo | Cat. | Registro ICA | Dosis/ha | Unidad | Carencia (d) | Reingreso (h) | Precio/unidad (COP) |
|---|---|---|---|---|---|---|---|---|---|
| NitroCampo 46 | fertilizante | Urea 46 % N | IV | ICA-DEMO-1001 | 150 | kg | 0 | 0 | 3800 |
| FosfoArranque | fertilizante | Fosfato diamónico 18-46-0 | IV | ICA-DEMO-1002 | 100 | kg | 0 | 0 | 4500 |
| CalViva | enmienda | Cal dolomita (CaCO₃·MgCO₃) | IV | ICA-DEMO-1003 | 500 | kg | 0 | 0 | 600 |
| BioLarva WP | bioinsumo | *Bacillus thuringiensis* var. kurstaki | IV | ICA-DEMO-2001 | 0.5 | kg | 0 | 4 | 95000 |
| RoyaStop 250 EC | fungicida | Triazol 250 g/L | III | ICA-DEMO-3001 | 0.4 | L | 30 | 24 | 160000 |
| TizónFin 80 WP | fungicida | Ditiocarbamato 800 g/kg | III | ICA-DEMO-3002 | 2.5 | kg | 14 | 24 | 38000 |

In [ ]:
# 2.3 Registrar los productos del anexo (los tres primeros ya están; ustedes completan el resto en el RETO 2)
sql('''
INSERT INTO producto (nombre_comercial, tipo, ingrediente_activo, categoria_tox, registro_ica,
                      dosis_ha, unidad, carencia_dias, reingreso_horas, precio_unidad) VALUES
 ('NitroCampo 46', 'fertilizante', 'Urea 46 % N',                'IV', 'ICA-DEMO-1001', 150, 'kg', 0, 0, 3800),
 ('FosfoArranque', 'fertilizante', 'Fosfato diamónico 18-46-0',  'IV', 'ICA-DEMO-1002', 100, 'kg', 0, 0, 4500),
 ('CalViva',       'enmienda',     'Cal dolomita',               'IV', 'ICA-DEMO-1003', 500, 'kg', 0, 0,  600);
SELECT id_producto, nombre_comercial, tipo, categoria_tox, dosis_ha, unidad FROM producto;
''')

### ✏️ RETO 2A · Registrar los tres productos faltantes
Copien con cuidado los datos de la tabla de anexos. Un error en la categoría toxicológica o en el periodo de carencia es un error de seguridad, no solo de digitación.

In [ ]:
# ✏️ RETO 2A
sql('''
-- INSERT INTO producto (...) VALUES (...), (...), (...);
SELECT nombre_comercial, tipo, categoria_tox, carencia_dias, reingreso_horas FROM producto;
''')

In [ ]:
# 2.4 Registrar aplicaciones (uso de llaves foráneas)
sql('''
INSERT INTO aplicacion (id_lote, id_producto, fecha, cantidad, responsable) VALUES
 (1, 1, '2026-04-02', 520, 'Jorge Patiño'),
 (1, 2, '2026-03-02', 350, 'Jorge Patiño'),
 (4, 1, '2026-05-25', 360, 'Luz Marina Ortiz'),
 (5, 3, '2026-05-20', 2000,'Ramiro Suárez'),
 (3, 1, '2026-02-11', 1000,'Luz Marina Ortiz');
SELECT * FROM aplicacion;
''')

In [ ]:
# 2.5 Estas sentencias DEBEN fallar. Expliquen cada error en las preguntas de reflexión.
sql("INSERT INTO aplicacion (id_lote, id_producto, fecha, cantidad, responsable) VALUES (99, 1, '2026-06-01', 10, 'Prueba')")
sql("DELETE FROM finca WHERE id_finca = 1")

### ✏️ RETO 2B
1. Registren una aplicación de **RoyaStop 250 EC** en el lote de café `POR-L01` usando la dosis del anexo (cantidad = dosis × área). Calculen la cantidad con SQL usando una subconsulta en lugar de escribir el número a mano.
2. Escriban la consulta que muestre, para cada aplicación, la **fecha mínima de cosecha permitida** (`fecha + carencia_dias`) usando `DATE_ADD`.

In [ ]:
# ✏️ RETO 2B
sql('''
-- 1) INSERT INTO aplicacion (id_lote, id_producto, fecha, cantidad, responsable)
--    SELECT l.id_lote, p.id_producto, '2026-10-14', p.dosis_ha * l.area_ha, 'Su nombre'
--    FROM lote l, producto p WHERE ... ;

-- 2) SELECT ..., DATE_ADD(a.fecha, INTERVAL p.carencia_dias DAY) AS cosecha_desde FROM ...
''')

### Preguntas de reflexión 2
1. ¿Qué error dio el `INSERT` con `id_lote = 99`? ¿Qué pasaría en una página web si no existiera esa llave foránea?
2. ¿Por qué no se pudo borrar la finca 1? ¿En qué caso del negocio sí usarían `ON DELETE CASCADE`?
3. ¿Por qué `registro_ica` es `UNIQUE`? ¿Qué riesgo hay si dos productos distintos quedan con el mismo registro?
4. Un operario entra al lote 10 horas después de aplicar RoyaStop. ¿Qué dato de la base de datos debió impedirlo y cómo lo mostrarían en la aplicación web?

*Respuestas:* (escriban aquí)

---
## Ejercicio 3 · Consultas para tomar decisiones

### Objetivo
Responder preguntas reales del agrónomo combinando varias tablas.

### Fundamento
- `SELECT … FROM … WHERE …` filtra filas.
- `JOIN … ON …` combina tablas a través de sus llaves. `INNER JOIN` muestra solo lo que coincide; `LEFT JOIN` conserva todas las filas de la tabla izquierda aunque no tengan pareja.
- `GROUP BY` agrupa filas y permite funciones de agregado: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`.
- `HAVING` filtra **grupos** (después de agrupar); `WHERE` filtra **filas** (antes de agrupar).

Orden lógico en que el motor evalúa una consulta:
`FROM → JOIN → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT`

In [ ]:
# 3.1 Lotes con el nombre de su finca (INNER JOIN)
sql('''
USE agro_campo;
SELECT f.nombre AS finca, l.codigo, l.cultivo, l.area_ha
FROM lote l
JOIN finca f ON f.id_finca = l.id_finca
ORDER BY f.nombre, l.codigo;
''')

In [ ]:
# 3.2 Costo de insumos por lote (JOIN de tres tablas + agregado)
sql('''
SELECT l.codigo, l.cultivo,
       COUNT(a.id_aplicacion)                        AS aplicaciones,
       ROUND(SUM(a.cantidad * p.precio_unidad), 0)   AS costo_total_cop,
       ROUND(SUM(a.cantidad * p.precio_unidad) / l.area_ha, 0) AS costo_por_ha
FROM lote l
JOIN aplicacion a ON a.id_lote = l.id_lote
JOIN producto  p ON p.id_producto = a.id_producto
GROUP BY l.id_lote, l.codigo, l.cultivo, l.area_ha
ORDER BY costo_total_cop DESC;
''')

In [ ]:
# 3.3 LEFT JOIN: lotes que todavía NO han recibido ninguna aplicación
sql('''
SELECT l.codigo, l.cultivo
FROM lote l
LEFT JOIN aplicacion a ON a.id_lote = l.id_lote
WHERE a.id_aplicacion IS NULL;
''')

In [ ]:
# 3.4 Comparar dosis aplicada vs dosis recomendada por el anexo
sql('''
SELECT l.codigo, p.nombre_comercial,
       ROUND(a.cantidad / l.area_ha, 1) AS dosis_aplicada_ha,
       p.dosis_ha                       AS dosis_anexo_ha,
       CASE
         WHEN a.cantidad / l.area_ha > p.dosis_ha * 1.10 THEN 'SOBREDOSIS'
         WHEN a.cantidad / l.area_ha < p.dosis_ha * 0.90 THEN 'SUBDOSIS'
         ELSE 'OK'
       END AS evaluacion
FROM aplicacion a
JOIN lote l     ON l.id_lote = a.id_lote
JOIN producto p ON p.id_producto = a.id_producto;
''')

### ✏️ RETO 3
Escriban una consulta para cada pregunta del agrónomo:
1. Área total sembrada por **cultivo**, de mayor a menor.
2. Fincas cuya área total sembrada **supera 5 ha** (usen `HAVING`).
3. Productos que **nunca** se han aplicado.
4. El responsable que más kilogramos de fertilizante ha aplicado (pista: `WHERE p.tipo = 'fertilizante'`, `ORDER BY … LIMIT 1`).

In [ ]:
# ✏️ RETO 3
sql('''
-- 1)
-- 2)
-- 3)
-- 4)
''')

### Preguntas de reflexión 3
1. En la consulta 3.3, ¿qué pasaría si se cambia `LEFT JOIN` por `JOIN`? Expliquen por qué.
2. ¿Por qué en la consulta 3.2 el `GROUP BY` incluye `l.codigo` y `l.cultivo` además de `l.id_lote`?
3. Si la página web muestra un tablero con el costo por hectárea, ¿conviene calcularlo en SQL o en JavaScript del navegador? Argumenten pensando en cantidad de datos transferidos y seguridad.

*Respuestas:* (escriban aquí)

---
## Ejercicio 4 · Normalización de una planilla de campo

### Objetivo
Detectar los problemas de una tabla "tipo Excel" y llevarla a **Tercera Forma Normal (3FN)**.

### Fundamento
Muchas fincas llevan sus registros en una sola planilla. Al pasarla tal cual a una tabla aparecen **anomalías**:

- **De actualización:** el teléfono de un operario está repetido en 20 filas; si cambia, hay que cambiarlo en todas (y alguna se olvida).
- **De inserción:** no se puede registrar un operario nuevo hasta que haga su primera aplicación.
- **De borrado:** al borrar la única aplicación de un producto se pierde toda la información del producto.

**Formas normales:**
- **1FN:** valores atómicos, sin listas en una celda ni grupos repetidos.
- **2FN:** 1FN + todo atributo depende de **toda** la llave (no de una parte).
- **3FN:** 2FN + ningún atributo depende de otro atributo que no es llave (sin **dependencias transitivas**). Ejemplo: `documento_operario → telefono_operario`.

In [ ]:
# 4.1 La planilla tal como llega del campo
sql('''
USE agro_campo;
DROP TABLE IF EXISTS planilla_campo;
CREATE TABLE planilla_campo (
    fila               INT AUTO_INCREMENT PRIMARY KEY,
    fecha              DATE,
    codigo_lote        VARCHAR(10),
    cultivo            VARCHAR(20),
    producto           VARCHAR(80),
    categoria_tox      VARCHAR(3),
    cantidad           DECIMAL(8,2),
    documento_operario VARCHAR(15),
    nombre_operario    VARCHAR(80),
    telefono_operario  VARCHAR(15),
    observaciones      VARCHAR(200)
);
INSERT INTO planilla_campo (fecha, codigo_lote, cultivo, producto, categoria_tox, cantidad,
                            documento_operario, nombre_operario, telefono_operario, observaciones) VALUES
 ('2026-06-03','ESP-L01','maíz',  'NitroCampo 46','IV', 500,'1012','Jorge Patiño',    '3101112233','lluvia en la tarde'),
 ('2026-06-03','ESP-L02','tomate','TizónFin 80 WP','III',  3,'1012','Jorge Patiño',    '3101112233','manchas en hojas bajeras; tizón'),
 ('2026-06-10','POR-L02','maíz',  'BioLarva WP',  'IV',  1.2,'2045','Luz Marina Ortiz','3125556677','cogollero en 30% de plantas'),
 ('2026-06-12','POR-L01','café',  'RoyaStop 250 EC','III',2.7,'2045','Luz M. Ortiz',   '3125556677','roya; reingreso 24h'),
 ('2026-06-15','VRO-L01','papa',  'TizónFin 80 WP','III', 10,'3090','Ramiro Suárez',   '3009998877','tizón tardío'),
 ('2026-06-20','ESP-L01','maíz',  'NitroCampo 46','IV', 120,'1012','Jorge Patiño',    '3159990000','segunda fertilización');
SELECT * FROM planilla_campo;
''')

Observen la última fila: **el mismo operario (1012) aparece con otro teléfono**, y la fila 4 escribe el nombre de Luz Marina de otra forma. Esas son anomalías de actualización ya ocurridas.

In [ ]:
# 4.2 Diagnóstico con SQL: ¿qué operarios tienen datos contradictorios?
sql('''
SELECT documento_operario,
       COUNT(DISTINCT nombre_operario)   AS nombres_distintos,
       COUNT(DISTINCT telefono_operario) AS telefonos_distintos
FROM planilla_campo
GROUP BY documento_operario
HAVING nombres_distintos > 1 OR telefonos_distintos > 1;
''')

In [ ]:
# 4.3 Normalización: el operario pasa a su propia tabla (elimina la dependencia transitiva)
sql('''
DROP TABLE IF EXISTS operario_n;
CREATE TABLE operario_n (
    documento       VARCHAR(15) PRIMARY KEY,
    nombre_completo VARCHAR(80) NOT NULL,
    telefono        VARCHAR(15)
);
-- Tomamos el dato más reciente de cada operario como versión correcta
INSERT INTO operario_n (documento, nombre_completo, telefono)
SELECT p.documento_operario, p.nombre_operario, p.telefono_operario
FROM planilla_campo p
JOIN (SELECT documento_operario, MAX(fecha) AS ultima
      FROM planilla_campo GROUP BY documento_operario) u
  ON u.documento_operario = p.documento_operario AND u.ultima = p.fecha;
SELECT * FROM operario_n;
''')

### ✏️ RETO 4
1. Creen la tabla `aplicacion_n` en 3FN con: `id`, `fecha`, `id_lote` (FK a `lote`), `id_producto` (FK a `producto`), `cantidad`, `documento_operario` (FK a `operario_n`), `observaciones`.
2. Migren los datos de `planilla_campo` a `aplicacion_n` con **un solo** `INSERT … SELECT` que haga `JOIN` con `lote` (por código) y con `producto` (por nombre). Si algún producto de la planilla no existe en `producto`, regístrenlo primero (Reto 2A).
3. Escriban la consulta que reconstruya la planilla original a partir de las tablas normalizadas (pista: es un `SELECT` con tres `JOIN`).

In [ ]:
# ✏️ RETO 4
sql('''
-- 1) CREATE TABLE aplicacion_n ( ... );

-- 2) INSERT INTO aplicacion_n (...) SELECT ... FROM planilla_campo pc JOIN lote l ON ... JOIN producto p ON ...;

-- 3) SELECT ... FROM aplicacion_n a JOIN lote l ... JOIN producto p ... JOIN operario_n o ...;
''')

### Preguntas de reflexión 4
1. Identifiquen en la planilla original un ejemplo concreto de cada anomalía: actualización, inserción y borrado.
2. La columna `categoria_tox` de la planilla, ¿de qué depende realmente: de la aplicación o del producto? ¿Qué forma normal violaba?
3. La columna `observaciones` contiene texto como "cogollero en 30% de plantas". ¿Cumple 1FN? ¿Cómo la rediseñarían para poder consultar cuántas plantas tenían plaga? (Esto conecta con el Ejercicio 5.)
4. ¿Cuándo es aceptable **desnormalizar** a propósito? Piensen en un reporte de una página web que se consulta miles de veces al día.

*Respuestas:* (escriban aquí)

---
## Ejercicio 5 · Visión artificial con YOLO + MySQL

### Objetivo
Guardar en MySQL las detecciones de un modelo **YOLO** y convertirlas en información para decidir qué producto aplicar.

### Fundamento: ¿qué es YOLO?
**YOLO (You Only Look Once)** es una familia de redes neuronales de **detección de objetos**. A diferencia de un clasificador (que dice *"en esta foto hay roya"*), un detector responde **qué hay y dónde**: entrega una lista de cajas.

Cada detección tiene:
| Campo | Significado |
|---|---|
| `clase` | Qué objeto es (por ejemplo `gusano_cogollero`, `roya`, `maleza`) |
| `confianza` | Probabilidad estimada (0 a 1) de que la caja sea correcta |
| `x, y, w, h` | Centro, ancho y alto de la caja, normalizados entre 0 y 1 respecto a la imagen |

Ideas clave:
- YOLO procesa **toda la imagen en una sola pasada** por la red; por eso es rápido y sirve para video de drones en tiempo real.
- Produce muchas cajas candidatas. Se filtran con un **umbral de confianza** y luego con **NMS** (*Non-Maximum Suppression*), que elimina cajas repetidas sobre el mismo objeto usando el **IoU** (*Intersection over Union*: área de intersección ÷ área de unión de dos cajas).
- Un modelo preentrenado (COCO) conoce 80 clases generales (persona, carro, banano, brócoli…). Para plagas del cultivo se hace **entrenamiento personalizado** con fotos etiquetadas del campo.

### Por qué importa la base de datos
Un dron puede generar miles de detecciones por hectárea. Sin una base de datos bien diseñada no se puede responder: *¿qué lotes superan el umbral de daño?, ¿la infestación sube o baja?, ¿qué producto y cuánta cantidad aplico?*

Modelo: **LOTE (1) ──< IMAGEN (1) ──< DETECCION**, y **CLASE_OBJETIVO >── PRODUCTO** para la recomendación.

In [ ]:
# 5.1 Tablas para visión artificial
sql('''
USE agro_campo;
CREATE TABLE imagen (
    id_imagen    INT AUTO_INCREMENT PRIMARY KEY,
    id_lote      INT NOT NULL,
    fecha_hora   DATETIME NOT NULL,
    fuente       ENUM('dron','celular','cámara fija') NOT NULL,
    archivo      VARCHAR(200) NOT NULL,
    modelo       VARCHAR(40) NOT NULL,
    FOREIGN KEY (id_lote) REFERENCES lote(id_lote)
);

CREATE TABLE deteccion (
    id_deteccion INT AUTO_INCREMENT PRIMARY KEY,
    id_imagen    INT NOT NULL,
    clase        VARCHAR(40) NOT NULL,
    confianza    DECIMAL(4,3) NOT NULL CHECK (confianza BETWEEN 0 AND 1),
    x  DECIMAL(5,4) NOT NULL, y DECIMAL(5,4) NOT NULL,
    w  DECIMAL(5,4) NOT NULL, h DECIMAL(5,4) NOT NULL,
    FOREIGN KEY (id_imagen) REFERENCES imagen(id_imagen) ON DELETE CASCADE,
    INDEX idx_clase_conf (clase, confianza)
);

CREATE TABLE clase_objetivo (
    clase            VARCHAR(40) PRIMARY KEY,
    descripcion      VARCHAR(150) NOT NULL,
    umbral_accion    DECIMAL(4,3) NOT NULL,   -- confianza mínima para contar la detección
    max_por_imagen   INT NOT NULL,            -- umbral de daño económico (didáctico)
    id_producto      INT NULL,
    FOREIGN KEY (id_producto) REFERENCES producto(id_producto)
);
''')

**Parte A · YOLO real con una foto.** Usaremos la librería `ultralytics` con un modelo preentrenado en COCO. Pueden subir una foto propia (frutas, hortalizas, una planta en maceta) o usar la imagen de ejemplo.

In [ ]:
# 5.2 Instalar YOLO y detectar objetos en una imagen
!pip -q install ultralytics
from ultralytics import YOLO
import os

modelo = YOLO("yolo11n.pt")   # versión 'nano', la más liviana. También sirve "yolov8n.pt"

SUBIR_FOTO = False            # cámbienlo a True para subir su propia foto
if SUBIR_FOTO:
    from google.colab import files
    archivo = list(files.upload().keys())[0]
else:
    archivo = "https://ultralytics.com/images/bus.jpg"

resultado = modelo(archivo, conf=0.25)[0]
resultado.show()   # dibuja las cajas sobre la imagen

for caja in resultado.boxes:
    clase = modelo.names[int(caja.cls)]
    conf  = float(caja.conf)
    x, y, w, h = [float(v) for v in caja.xywhn[0]]
    print(f"{clase:15s} conf={conf:.2f}  centro=({x:.2f},{y:.2f})  tamaño=({w:.2f}x{h:.2f})")

In [ ]:
# 5.3 Guardar esas detecciones en MySQL (sentencias parametrizadas: así se evita la inyección SQL)
CON.ping(reconnect=True)
with CON.cursor() as cur:
    cur.execute("INSERT INTO imagen (id_lote, fecha_hora, fuente, archivo, modelo) VALUES (%s, NOW(), %s, %s, %s)",
                (2, "celular", os.path.basename(str(archivo)), "yolo11n-coco"))
    id_img = cur.lastrowid
    filas = [(id_img, modelo.names[int(c.cls)], round(float(c.conf), 3),
              *[round(float(v), 4) for v in c.xywhn[0]]) for c in resultado.boxes]
    cur.executemany("INSERT INTO deteccion (id_imagen, clase, confianza, x, y, w, h) VALUES (%s,%s,%s,%s,%s,%s,%s)", filas)
print(f"Imagen {id_img} guardada con {len(filas)} detecciones")
sql(f"SELECT clase, COUNT(*) AS n, ROUND(AVG(confianza),2) AS conf_media FROM deteccion WHERE id_imagen = {id_img} GROUP BY clase")

**Parte B · Modelo personalizado del cultivo (simulado).** Entrenar un YOLO para plagas toma horas y requiere miles de fotos etiquetadas. Para concentrarnos en la base de datos, la siguiente celda **simula** la salida que daría un modelo entrenado con las clases `gusano_cogollero`, `roya`, `tizon`, `maleza` y `planta_sana` sobre 30 imágenes de dron. El formato de los datos es idéntico al de la Parte A.

In [ ]:
# 5.4 Catálogo de clases y su producto recomendado
sql('''
INSERT INTO clase_objetivo VALUES
 ('gusano_cogollero','Larva de Spodoptera frugiperda en el cogollo del maíz', 0.50, 3,
     (SELECT id_producto FROM producto WHERE nombre_comercial = 'BioLarva WP')),
 ('roya',            'Pústulas anaranjadas en el envés de la hoja de café', 0.55, 4,
     (SELECT id_producto FROM producto WHERE nombre_comercial = 'RoyaStop 250 EC')),
 ('tizon',           'Lesiones necróticas en hojas de papa y tomate',        0.55, 2,
     (SELECT id_producto FROM producto WHERE nombre_comercial = 'TizónFin 80 WP')),
 ('maleza',          'Planta arvense compitiendo con el cultivo',            0.45, 8, NULL),
 ('planta_sana',     'Planta del cultivo sin síntomas',                      0.40, 999, NULL);
SELECT c.clase, c.umbral_accion, c.max_por_imagen, p.nombre_comercial AS producto
FROM clase_objetivo c LEFT JOIN producto p ON p.id_producto = c.id_producto;
''')
# Si 'producto' sale NULL en roya/cogollero/tizón, completen primero el RETO 2A.

In [ ]:
# 5.5 Simular 30 vuelos de dron y guardar las detecciones
import random, datetime
random.seed(2026)
plagas_por_cultivo = {"maíz": ["gusano_cogollero", "maleza"], "café": ["roya", "maleza"],
                      "papa": ["tizon", "maleza"], "tomate": ["tizon", "maleza"]}
lotes = sql("SELECT id_lote, cultivo FROM lote", mostrar=False).values.tolist()

CON.ping(reconnect=True)
with CON.cursor() as cur:
    for n in range(30):
        id_lote, cultivo = random.choice(lotes)
        fecha = datetime.datetime(2026, 9, 1) + datetime.timedelta(days=random.randint(0, 40), hours=random.randint(6, 16))
        cur.execute("INSERT INTO imagen (id_lote, fecha_hora, fuente, archivo, modelo) VALUES (%s,%s,'dron',%s,'yolo11n-agro-v1')",
                    (id_lote, fecha, f"vuelo_{n:03d}.jpg"))
        id_img = cur.lastrowid
        presion = random.random()                        # qué tan afectado está el lote ese día
        dets = []
        for _ in range(random.randint(6, 18)):
            clase = random.choice(plagas_por_cultivo[cultivo]) if random.random() < presion else "planta_sana"
            conf = round(min(0.99, max(0.05, random.gauss(0.68, 0.18))), 3)
            dets.append((id_img, clase, conf, round(random.uniform(.05,.95),4), round(random.uniform(.05,.95),4),
                         round(random.uniform(.03,.20),4), round(random.uniform(.03,.20),4)))
        cur.executemany("INSERT INTO deteccion (id_imagen, clase, confianza, x, y, w, h) VALUES (%s,%s,%s,%s,%s,%s,%s)", dets)

sql("SELECT modelo, COUNT(DISTINCT i.id_imagen) AS imagenes, COUNT(*) AS detecciones FROM imagen i JOIN deteccion d USING(id_imagen) GROUP BY modelo")

In [ ]:
# 5.6 Vista: detecciones válidas (superan el umbral de su clase) con lote y cultivo
sql('''
CREATE OR REPLACE VIEW v_deteccion_valida AS
SELECT d.id_deteccion, i.id_imagen, i.fecha_hora, l.codigo AS lote, l.cultivo, l.area_ha,
       d.clase, d.confianza
FROM deteccion d
JOIN imagen i         ON i.id_imagen = d.id_imagen
JOIN lote l           ON l.id_lote = i.id_lote
JOIN clase_objetivo c ON c.clase = d.clase
WHERE d.confianza >= c.umbral_accion;

-- ¿Cuántas detecciones de cada clase se descartaron por baja confianza?
SELECT d.clase,
       COUNT(*) AS total,
       SUM(d.confianza >= c.umbral_accion) AS validas,
       SUM(d.confianza <  c.umbral_accion) AS descartadas
FROM deteccion d JOIN clase_objetivo c ON c.clase = d.clase
GROUP BY d.clase;
''')

In [ ]:
# 5.7 De la detección a la decisión: imágenes que superan el umbral de daño y producto sugerido
sql('''
SELECT v.lote, v.cultivo, v.id_imagen, DATE(v.fecha_hora) AS fecha, v.clase,
       COUNT(*) AS detecciones, c.max_por_imagen AS umbral_dano,
       p.nombre_comercial AS producto_sugerido,
       ROUND(p.dosis_ha * v.area_ha, 2) AS cantidad_lote, p.unidad,
       p.carencia_dias, p.reingreso_horas
FROM v_deteccion_valida v
JOIN clase_objetivo c ON c.clase = v.clase
LEFT JOIN producto p  ON p.id_producto = c.id_producto
WHERE v.clase <> 'planta_sana'
GROUP BY v.lote, v.cultivo, v.area_ha, v.id_imagen, fecha, v.clase, c.max_por_imagen,
         p.nombre_comercial, p.dosis_ha, p.unidad, p.carencia_dias, p.reingreso_horas
HAVING COUNT(*) > c.max_por_imagen
ORDER BY detecciones DESC;
''')

In [ ]:
# 5.8 TRIGGER: crear una alerta automática cuando llega una detección de plaga con confianza >= 0.85
sql('''
CREATE TABLE IF NOT EXISTS alerta (
    id_alerta    INT AUTO_INCREMENT PRIMARY KEY,
    id_deteccion INT NOT NULL,
    mensaje      VARCHAR(200) NOT NULL,
    creada       DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (id_deteccion) REFERENCES deteccion(id_deteccion) ON DELETE CASCADE
);
DROP TRIGGER IF EXISTS trg_alerta_plaga;
''')
sql_bloque('''
CREATE TRIGGER trg_alerta_plaga AFTER INSERT ON deteccion
FOR EACH ROW
BEGIN
  IF NEW.clase IN ('gusano_cogollero','roya','tizon') AND NEW.confianza >= 0.85 THEN
    INSERT INTO alerta (id_deteccion, mensaje)
    VALUES (NEW.id_deteccion, CONCAT('Revisar ', NEW.clase, ' en imagen ', NEW.id_imagen,
                                     ' (confianza ', NEW.confianza, ')'));
  END IF;
END
''')
# Probar el trigger con una detección nueva
sql('''
INSERT INTO deteccion (id_imagen, clase, confianza, x, y, w, h) VALUES (2, 'roya', 0.910, 0.5, 0.5, 0.1, 0.1);
SELECT * FROM alerta;
''')

### ✏️ RETO 5
1. **Tendencia:** para cada lote, cuenten las detecciones válidas de plaga por **semana** (`YEARWEEK(fecha_hora)`). ¿Algún lote empeora semana a semana?
2. **Densidad:** calculen detecciones válidas de plaga **por hectárea** en cada lote.
3. **Calidad del modelo:** el umbral de confianza de `maleza` es 0.45. Escriban una consulta que muestre cuántas detecciones de maleza quedarían si el umbral fuera 0.30, 0.45, 0.60 y 0.75 (una fila por umbral; pista: `UNION ALL` o `SUM(confianza >= …)`).
4. **Diseño:** el próximo mes la empresa entrena `yolo11n-agro-v2`. Propongan cómo cambiar el modelo de datos para comparar ambas versiones sobre las mismas imágenes (escriban el `CREATE TABLE` o `ALTER TABLE`).

In [ ]:
# ✏️ RETO 5
sql('''
-- 1)
-- 2)
-- 3)
-- 4)
''')

### Preguntas de reflexión 5
1. ¿Qué diferencia hay entre **clasificación** y **detección de objetos**? ¿Por qué para contar gusanos por planta se necesita detección?
2. Si suben el umbral de confianza de 0.25 a 0.80, ¿qué pasa con los **falsos positivos** y con los **falsos negativos**? ¿Qué es más costoso para el agricultor en el caso del gusano cogollero?
3. ¿Qué es el **IoU** y para qué lo usa el algoritmo **NMS**? Den un ejemplo con dos cajas sobre la misma hoja con roya.
4. ¿Por qué las coordenadas se guardan **normalizadas** (0 a 1) y no en píxeles?
5. En la celda 5.3 se usaron sentencias **parametrizadas** (`%s`). ¿Qué ataque evitan y por qué es obligatorio usarlas en el backend de una página web?
6. Un modelo COCO detectó "broccoli" en un lote de papa. ¿Qué nos dice esto sobre usar modelos preentrenados fuera de su dominio?

*Respuestas:* (escriban aquí)

---
## Cierre · Conexión con desarrollo web

Lo que construyeron es el *backend de datos* de una aplicación web agrícola. Para llevarlo a una página:

```
[ Navegador: HTML + CSS + JS ] --fetch('/api/alertas')--> [ Backend PHP/Node/Python ] --SELECT * FROM alerta--> [ MySQL ]
                               <-------- JSON ------------                          <------ filas -----------
```

Ejemplo de cómo el backend convierte una consulta en JSON (lo que recibe `fetch` en JavaScript):

In [ ]:
import json
datos = sql("SELECT lote, clase, COUNT(*) AS n FROM v_deteccion_valida WHERE clase <> 'planta_sana' GROUP BY lote, clase ORDER BY n DESC LIMIT 5", mostrar=False)
print(json.dumps(datos.to_dict(orient="records"), ensure_ascii=False, indent=2, default=str))

## Rúbrica de evaluación sugerida

| Criterio | Peso | Excelente | Aceptable | Por mejorar |
|---|---|---|---|---|
| Retos SQL (1 a 5) ejecutan sin error y responden la pregunta | 40 % | Todas correctas y con alias claros | 3 o 4 retos correctos | Menos de 3 |
| Uso correcto de restricciones e integridad | 15 % | Justifica cada PK, FK, CHECK | Las usa sin justificar | Omite restricciones |
| Normalización (Ejercicio 4) | 15 % | 3FN completa y reconstrucción correcta | 2FN | Tabla plana |
| Interpretación de YOLO + datos | 15 % | Relaciona umbral, IoU y decisión agronómica | Explica conceptos sin relacionarlos | Respuestas incompletas |
| Preguntas de reflexión | 15 % | Argumentadas con ejemplos del cultivo | Respuestas cortas correctas | Sin responder |

**Entrega:** `Archivo → Descargar → Descargar .ipynb` y subirlo a la plataforma del curso.